# Week 13, Day 6: Forge: Week 13 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w013/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
k = 100
print("1. ((sqrt(k) - 1) / (sqrt(k) + 1))^2 =", round(((np.sqrt(k) - 1) / (np.sqrt(k) + 1)) ** 2, 4))
print("2. every entry moves by lr against its sign: (+lr, -lr)")
print("3.", np.sign(-1.3) * max(abs(-1.3) - 0.5, 0))

In [ ]:
rng = np.random.default_rng(2018)
N, T = 50, 2000
prefs = np.linspace(0, np.pi, N, endpoint=False)
theta = rng.uniform(0, np.pi, T)
labels = (np.cos(2 * (theta - np.pi / 4)) > 0).astype(float)
counts = rng.poisson((10 + 20 * np.exp(2 * (np.cos(2 * (theta[:, None] - prefs[None])) - 1))) * 0.2)
tr, va = np.arange(1500), np.arange(1500, T)
def sigmoid(u): return 1 / (1 + np.exp(-u))
def xent(w, Z, y):
    p = np.clip(sigmoid(Z @ w), 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
def grad(w, Z, y): return Z.T @ (sigmoid(Z @ w) - y) / len(y)
Z = np.column_stack([counts, np.ones(T)])
Ztr, ytr, Zva, yva = Z[tr], labels[tr], Z[va], labels[va]
lr = 1.9 / np.linalg.eigvalsh(Ztr.T @ Ztr / (4 * len(tr))).max()

### Exercise: Part A: faster to fit, faster to overfit

Train for 1,000 steps with gradient descent (lr above), momentum (same lr, β = 0.9) and Adam (learning rate 0.03), recording the validation loss after each step. For each, store in `results[name]` the final training loss under 'train', the list of validation losses under 'val', and the step with the lowest validation loss under 'best_step'.

In [ ]:
def train(kind, steps=1000):
    w, v, m, s = np.zeros(N + 1), np.zeros(N + 1), np.zeros(N + 1), np.zeros(N + 1)
    val = []
    for k in range(1, steps + 1):
        g = grad(w, Ztr, ytr)
        if kind == "GD":
            w = ...
        elif kind == "momentum":
            ...
        else:
            ...
        val.append(xent(w, Zva, yva))
    return {"train": xent(w, Ztr, ytr), "val": val, "best_step": int(np.argmin(val)) + 1}

results = {name: train(name) for name in ["GD", "momentum", "Adam"]}
for name, r in results.items():
    print(f"{name:9s} train {r['train']:.4f}  best validation {min(r['val']):.4f} at step {r['best_step']}  final validation {r['val'][-1]:.4f}")
for name, r in results.items():
    plt.semilogx(np.arange(1, 1001), r["val"], label=name)
plt.xlabel("step"); plt.ylabel("validation loss"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_train_speed():
    "momentum and Adam reach a lower training loss than gradient descent after 1000 steps"
    assert results["momentum"]["train"] < results["GD"]["train"] and results["Adam"]["train"] < results["GD"]["train"]

def test_best_early():
    "momentum and Adam reach their lowest validation loss more than ten times sooner than gradient descent"
    assert 10 * results["momentum"]["best_step"] < results["GD"]["best_step"]
    assert 10 * results["Adam"]["best_step"] < results["GD"]["best_step"]

def test_overfit():
    "after 1000 steps, Adam's validation loss is above its own minimum by more than 0.01"
    assert results["Adam"]["val"][-1] > min(results["Adam"]["val"]) + 0.01

test_train_speed()
test_best_early()
test_overfit()
print("All checks passed")

### Exercise: Part B: the curve must move

Draw 20,000 shade intensities (log-normal, mean of the log 0, sd 0.5) and 20,000 sunlight intensities (mean of the log 1). Build the CDF response for each world and compute the ten-level entropy of the shade curve in shade, the shade curve in sunlight, and the sunlight curve in sunlight.

In [ ]:
x_shade = rng.lognormal(0, 0.5, 20000)
x_sun = rng.lognormal(1, 0.5, 20000)

def entropy10(r):
    p = np.bincount(np.minimum((r * 10).astype(int), 9), minlength=10) / len(r)
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

def cdf_curve(samples):
    s = np.sort(samples)
    return lambda x: ...

F_shade, F_sun = cdf_curve(x_shade), cdf_curve(x_sun)
H_shade_shade = entropy10(F_shade(x_shade))
H_shade_sun = ...
H_sun_sun = ...
print(f"shade curve in shade {H_shade_shade:.2f} bits, in sunlight {H_shade_sun:.2f} bits; sunlight curve in sunlight {H_sun_sun:.2f} bits")

In [ ]:
import numpy as np
def test_matched():
    "each curve gives the maximum entropy, log2(10), in its own world"
    assert abs(H_shade_shade - np.log2(10)) < 0.01 and abs(H_sun_sun - np.log2(10)) < 0.01

def test_mismatch():
    "the shade curve loses more than one bit in sunlight"
    assert H_shade_sun < np.log2(10) - 1

test_matched()
test_mismatch()
print("All checks passed")

### Exercise: Part C: the lasso on held-out data

For each λ in `lams`, run ISTA (3,000 steps) on the training trials and record the validation mean squared error. Store the errors in `val_err`, the best penalty in `lam_best` and its weights in `w_best`, and report how many inputs it selects and how many of them are real.

In [ ]:
n, p, k = 80, 120, 8
A = rng.gamma(2, 1, (n, p)); A = A - A.mean(0)
w_true = np.zeros(p)
true_inputs = rng.choice(p, k, replace=False)
w_true[true_inputs] = rng.uniform(0.5, 1.5, k)
y = A @ w_true + rng.normal(size=n)
Av = rng.gamma(2, 1, (200, p)); Av = Av - Av.mean(0)
yv = Av @ w_true + rng.normal(size=200)

def soft(z, tau): return np.sign(z) * np.maximum(np.abs(z) - tau, 0)
def ista(A, y, lam, steps=3000):
    L = np.linalg.norm(A, 2) ** 2
    w = np.zeros(A.shape[1])
    for _ in range(steps):
        w = soft(w - A.T @ (A @ w - y) / L, lam / L)
    return w

lams = [0.1, 1, 3, 10, 30, 100]
fits = ...
val_err = ...
lam_best = ...
w_best = ...
for lam, w, e in zip(lams, fits, val_err):
    sel = np.flatnonzero(w)
    print(f"lambda {lam:6}: validation error {e:.2f}, selects {len(sel):3d}, of which real {len(set(sel) & set(true_inputs))}")

In [ ]:
import numpy as np
def test_choice():
    "lam_best is the penalty with the lowest validation error"
    assert lam_best == lams[int(np.argmin(val_err))]

def test_recall():
    "the chosen model keeps all 8 true inputs"
    assert set(true_inputs) <= set(np.flatnonzero(w_best))

def test_selective():
    "it selects fewer than a third of the 120 candidates"
    assert np.count_nonzero(w_best) < 40

test_choice()
test_recall()
test_selective()
print("All checks passed")